## Project: Deep learning model that can remove noise from images using an autoencoder on MNIST

In [1]:

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, BatchNormalization
from tensorflow.keras.models import Model

np.random.seed(123)
tf.random.set_seed(123)


In [2]:

(x_train,_),(x_test,_) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32")/255.0
x_test = x_test.astype("float32")/255.0

x_train = x_train.reshape((-1,28,28,1))
x_test = x_test.reshape((-1,28,28,1))

noise_factor = 0.40

train_noisy = np.clip(
    x_train + noise_factor*np.random.normal(loc=0.0, scale=1.0, size=x_train.shape),
    0.,1.)

test_noisy = np.clip(
    x_test + noise_factor*np.random.normal(loc=0.0, scale=1.0, size=x_test.shape),
    0.,1.)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


## Build Autoencoder

In [3]:

inp = Input(shape=(28,28,1))

x = Conv2D(32,3,activation="relu",padding="same")(inp)
x = BatchNormalization()(x)
x = MaxPooling2D(2,padding="same")(x)

x = Conv2D(64,3,activation="relu",padding="same")(x)
x = BatchNormalization()(x)
encoded = MaxPooling2D(2,padding="same")(x)

x = Conv2D(64,3,activation="relu",padding="same")(encoded)
x = UpSampling2D(2)(x)

x = Conv2D(32,3,activation="relu",padding="same")(x)
x = UpSampling2D(2)(x)

out = Conv2D(1,3,activation="sigmoid",padding="same")(x)

autoencoder = Model(inp,out)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

autoencoder.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 32)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 28, 28, 1)      │           289 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,881 (292.50 KB)

 Trainable params: 74,689 (291.75 KB)

 Non-trainable params: 192 (768.00 B)

## Train

In [5]:

history = autoencoder.fit(
    train_noisy,
    x_train,
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(test_noisy, x_test),
    verbose=1
)


Epoch 1/20
 26/469 ━━━━━━━━━━━━━━━━━━━━ 4:05 554ms/step - accuracy: 0.8150 - loss: 0.0872

KeyboardInterrupt: 

In [ ]:

plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()


## Denoise Test Images

In [ ]:

pred = autoencoder.predict(test_noisy)

print("Prediction range:", pred.min(), pred.max())

n = 6
plt.figure(figsize=(10,8))

for i in range(n):

    plt.subplot(n,3,3*i+1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    plt.subplot(n,3,3*i+2)
    plt.imshow(test_noisy[i].squeeze(), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    plt.subplot(n,3,3*i+3)
    plt.imshow(pred[i].squeeze(), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")

plt.tight_layout()
plt.show()
